In [1]:
%load_ext watermark

In [2]:
%watermark

Last updated: 2025-10-13T22:49:51.156195+00:00

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.6.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 6.11.0-1016-nvidia
Machine     : aarch64
Processor   : aarch64
CPU cores   : 20
Architecture: 64bit



In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import optuna
import gc
import logging

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
%watermark --iversions

numpy  : 2.2.6
sklearn: 1.7.2
logging: 0.5.1.2
xgboost: 3.0.5
pandas : 2.3.3
optuna : 4.5.0



In [6]:
%%time
train_folds = []
val_folds = []
train_ys = []
val_ys = []

for i in range(5):
    print(f'Loading fold {i}')
    train_fold = pd.read_csv(f'../input/xgtrain_fold_{i}_l.csv.gz')

    
    val_fold = pd.read_csv(f'../input/xgval_fold_{i}_l.csv.gz')

    
    
    train_y = train_fold['target']
    train_fold = train_fold[train_fold.columns.difference(['target'])]
    
    val_y = val_fold['target']
    val_fold = val_fold[val_fold.columns.difference(['target'])]
    
    train_folds.append(train_fold)
    val_folds.append(val_fold)
    
    train_ys.append(train_y)
    val_ys.append(val_y)

Loading fold 0
Loading fold 1
Loading fold 2
Loading fold 3
Loading fold 4
CPU times: user 1.92 s, sys: 178 ms, total: 2.1 s
Wall time: 2.1 s


In [7]:
train = pd.read_csv('../input/train.csv.zip')

shift = 200

target0 = train['loss'].values
target = np.log(target0+shift)

In [8]:
train_oof = np.zeros((target.shape[0],))

num_round = 1000

def objective(trial):
        
    params = {
        'objective': 'reg:squarederror', 
        'base_score':7.76,
        'tree_method':'hist',  # 'gpu_hist','hist'
        'lambda': trial.suggest_float('lambda',1e-3,10.0, log=True),
        'alpha': trial.suggest_float('alpha',1e-3,10.0, log=True),
        'gamma': trial.suggest_float('gamma',1e-3,10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3,1.0),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'learning_rate': trial.suggest_float('learning_rate', 0.001,0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_child_weight': trial.suggest_int('min_child_weight', 1,300),
        'eval_metric': trial.suggest_categorical('eval_metric',['rmse']),

    }

    kf = KFold(5, shuffle=True, random_state=137)

    for i, (train_index, val_index) in enumerate(kf.split(train,target)):
        dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
        dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
        output = xgb.train(params, dtrain, num_round)
        #booster = output['booster']  # booster is the trained model
        #booster.set_param({'predictor': 'gpu_predictor'})
        predictions = output.predict(dval)
        train_oof[val_index] = np.exp(predictions) - shift
        del dtrain, dval, output
        gc.collect()
        gc.collect()

    mae = mean_absolute_error(target0, train_oof)
    
    return mae

In [9]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # Setup the root logger.
logger.addHandler(logging.FileHandler("optuna_xgb_output_l_4_DGX_Spark.log", mode="w"))

optuna.logging.enable_propagation()  # Propagate logs to the root logger.
optuna.logging.disable_default_handler()  # Stop showing logs in sys.stderr.

study = optuna.create_study(storage="sqlite:///xgb_optuna_allstate_l_4_DGX_Spark.db", study_name="five_fold_optuna_xgb_l_4", direction='minimize')

In [10]:
%%time
logger.info("Start optimization.")
study.optimize(objective, n_trials=3)

CPU times: user 25min 10s, sys: 20 s, total: 25min 30s
Wall time: 1min 34s


In [11]:
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.head()

,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1164.676148,0.011397,0.311138,rmse,4.510847,0.002734,0.030482,12,75,0.442689,COMPLETE
1,1,1142.711000,0.016854,0.760312,rmse,0.007938,0.152840,0.028536,7,151,0.407241,COMPLETE
2,2,1242.501987,0.053226,0.729803,rmse,2.594380,1.502792,0.002134,20,239,0.814082,COMPLETE


In [12]:
%%time
study.optimize(objective, n_trials=5)
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.to_csv('optuna_xgb_output_l_4_DGX_Spark.csv', index=False)
df

CPU times: user 1h 7min 38s, sys: 57.7 s, total: 1h 8min 35s
Wall time: 4min 14s


,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1164.676148,0.011397,0.311138,rmse,4.510847,0.002734,0.030482,12,75,0.442689,COMPLETE
1,1,1142.711000,0.016854,0.760312,rmse,0.007938,0.152840,0.028536,7,151,0.407241,COMPLETE
2,2,1242.501987,0.053226,0.729803,rmse,2.594380,1.502792,0.002134,20,239,0.814082,COMPLETE
3,3,1311.220965,1.534348,0.819081,rmse,0.071371,0.004626,0.001603,9,234,0.621605,COMPLETE
4,4,1172.817944,0.005848,0.890555,rmse,0.047285,0.001768,0.003585,22,226,0.954096,COMPLETE
5,5,1174.647968,0.028054,0.981314,rmse,0.056058,0.013156,0.004020,14,265,0.567921,COMPLETE
6,6,1145.982810,4.600571,0.914284,rmse,0.475847,0.024694,0.096164,15,259,0.972300,COMPLETE
7,7,1175.546863,0.087963,0.517328,rmse,0.008283,0.576797,0.058724,20,22,0.967603,COMPLETE


In [13]:
%%time
study.optimize(objective, n_trials=100)
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.to_csv('optuna_xgb_output_l_4_DGX_Spark.csv', index=False)
df.head(20)

CPU times: user 19h 34min 2s, sys: 13min 37s, total: 19h 47min 39s
Wall time: 1h 10min 42s


,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1164.676148,0.011397,0.311138,rmse,4.510847,0.002734,0.030482,12,75,0.442689,COMPLETE
1,1,1142.711000,0.016854,0.760312,rmse,0.007938,0.152840,0.028536,7,151,0.407241,COMPLETE
2,2,1242.501987,0.053226,0.729803,rmse,2.594380,1.502792,0.002134,20,239,0.814082,COMPLETE
3,3,1311.220965,1.534348,0.819081,rmse,0.071371,0.004626,0.001603,9,234,0.621605,COMPLETE
4,4,1172.817944,0.005848,0.890555,rmse,0.047285,0.001768,0.003585,22,226,0.954096,COMPLETE
5,5,1174.647968,0.028054,0.981314,rmse,0.056058,0.013156,0.004020,14,265,0.567921,COMPLETE
6,6,1145.982810,4.600571,0.914284,rmse,0.475847,0.024694,0.096164,15,259,0.972300,COMPLETE
7,7,1175.546863,0.087963,0.517328,rmse,0.008283,0.576797,0.058724,20,22,0.967603,COMPLETE
8,8,1137.010660,0.880404,0.309526,rmse,0.053779,5.356027,0.023779,11,296,0.877314,COMPLETE
9,9,1138.956837,0.020468,0.436963,rmse,0.009963,1.115986,0.019261,14,154,0.492666,COMPLETE


In [14]:
df

,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1164.676148,0.011397,0.311138,rmse,4.510847,0.002734,0.030482,12,75,0.442689,COMPLETE
1,1,1142.711000,0.016854,0.760312,rmse,0.007938,0.152840,0.028536,7,151,0.407241,COMPLETE
2,2,1242.501987,0.053226,0.729803,rmse,2.594380,1.502792,0.002134,20,239,0.814082,COMPLETE
3,3,1311.220965,1.534348,0.819081,rmse,0.071371,0.004626,0.001603,9,234,0.621605,COMPLETE
4,4,1172.817944,0.005848,0.890555,rmse,0.047285,0.001768,0.003585,22,226,0.954096,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...
103,103,1137.014650,0.002456,0.330555,rmse,0.007381,0.020099,0.028990,15,183,0.999572,COMPLETE
104,104,1135.901979,0.004090,0.305844,rmse,0.003656,0.007767,0.023023,15,202,0.987481,COMPLETE
105,105,1139.128251,0.004433,0.316974,rmse,0.003706,0.007514,0.035297,16,214,0.961657,COMPLETE
106,106,1138.370557,0.002071,0.348101,rmse,0.006129,0.021884,0.024237,13,200,0.672815,COMPLETE


In [15]:
df.value.min()

np.float64(1135.181651181601)

In [16]:
study.best_params

{'lambda': 0.022332889731473144,
 'alpha': 0.002573028546712259,
 'gamma': 0.0035388882913643515,
 'colsample_bytree': 0.3633701468331348,
 'subsample': 0.9826381900397495,
 'learning_rate': 0.017385196065109947,
 'max_depth': 15,
 'min_child_weight': 205,
 'eval_metric': 'rmse'}

In [ ]:
test = pd.read_csv(f'../input/xg')

In [27]:
%%time
train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

0
1
2
3
4
CPU times: user 2min 26s, sys: 1min 15s, total: 3min 42s
Wall time: 33.3 s


In [28]:
mean_absolute_error(target0, train_oof)

1135.6467608616847

In [30]:
%%time
num_round = 2500


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.742359873487
CPU times: user 5min 57s, sys: 3min 9s, total: 9min 7s
Wall time: 1min 21s


In [31]:
%%time
num_round = 2700


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.6704479178497
CPU times: user 6min 18s, sys: 3min 24s, total: 9min 43s
Wall time: 1min 27s


In [32]:
%%time
num_round = 3000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.6667403679148
CPU times: user 6min 55s, sys: 3min 47s, total: 10min 43s
Wall time: 1min 35s


In [33]:
%%time
num_round = 6000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.005

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.3884663044919
CPU times: user 13min 47s, sys: 7min 35s, total: 21min 23s
Wall time: 3min 11s


In [35]:
%%time
num_round = 5000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.005

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.4089241056433
CPU times: user 11min 46s, sys: 6min 19s, total: 18min 5s
Wall time: 2min 42s


In [36]:
x_test = pd.read_csv('../input/x_test_l.csv')

In [39]:
np.exp(output.predict(xgb.DMatrix(x_test))) - shift

array([2802.0037, 4143.8706, 3017.079 , ..., 4714.319 , 1551.7915,
       2243.5032], shape=(125546,), dtype=float32)